# Creating Custom Instrument Profiles in PyTestLab

A local YAML profile can map a device's supported commands to PyTestLab's typed
driver. Create a small power-supply profile and test it entirely in simulation.

## 1. Create a local profile

A profile declares its identity, the device type, channel limits and SCPI mappings.
The simulation section supplies local command responses and state updates.

Create the example in a temporary directory so it does not overwrite a local
profile or alter PyTestLab's built-in profiles.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

scratch = TemporaryDirectory(prefix="pytestlab-profile-")
profile_path = Path(scratch.name) / "my_psu_profile.yaml"
profile_path.write_text(r"""
manufacturer: MyBrand
model: PSU-101
device_type: power_supply
role: stimulus
channels:
  - channel_id: 1
    description: Single simulated output
    voltage_range: {min_val: 0, max_val: 6}
    current_limit_range: {min_val: 0, max_val: 1}
scpi:
  commands:
    set_voltage:
      template: "VOLT {voltage}, (@{channel})"
      validators:
        voltage: {min: 0, max: 6}
        channel: {min: 1, max: 1}
  queries:
    identify:
      template: "*IDN?"
      response: {type: str}
    measure_voltage:
      template: "MEAS:VOLT? (@{channel})"
      response: {type: float}
operation_options:
  operations:
    set_voltage:
      transport: {main: set_voltage}
      parameters:
        channel:
          kind: enum
          transport_identity: true
          options: [{value: 1, label: Channel 1}]
          bindings: [{alias: main, parameter: channel}]
        voltage:
          kind: range
          min: 0
          max: 6
          unit: V
          bindings: [{alias: main, parameter: voltage}]
    measure_voltage:
      transport: {main: measure_voltage}
      parameters:
        channel:
          kind: enum
          transport_identity: true
          options: [{value: 1, label: Channel 1}]
          bindings: [{alias: main, parameter: channel}]
simulation:
  initial_state:
    voltage: 0.0
  scpi:
    "*IDN?": "MyBrand,PSU-101,SIM,1.0"
    "SYST:ERR?": '+0,"No error"'
    'VOLT (.*), \(@1\)':
      set: {voltage: "$1"}
    'MEAS:VOLT\? \(@1\)':
      get: voltage
""".lstrip(), encoding="utf-8")

- `device_type` selects the typed power-supply driver.
- `channels` describes the supported channel and settings ranges.
- `scpi.commands` and `scpi.queries` map typed operations to transport commands.
- `operation_options` exposes typed channel and voltage arguments.
- `simulation` supplies a local fixture for this tutorial.

## 2. Define commands and queries

The driver renders the `set_voltage` template with the supplied channel and voltage.
The `measure_voltage` query declares that the response is a float.

The example deliberately implements only identification, voltage configuration, and
readback; it has no output or current-control commands.

## 3. Set up a simulation fixture

The simulator starts at zero volts. A matching `VOLT` command updates its voltage,
and the readback query returns that state.

## 4. Validate the assembled profile

PyTestLab validates the selected local YAML file when it builds the instrument.
A successful simulated interaction checks that the profile and simulator mappings
work together; it does not verify compatibility with physical hardware.

## 5. Load and use the local profile

Pass `simulate=True` to ensure the example uses the profile’s local simulator.

In [ ]:
from pytestlab import AutoInstrument

psu = AutoInstrument.from_config(profile_path, simulate=True)
try:
    identity = psu.id()
    assert "PSU-101" in identity
    psu.set_voltage(1, 5.0)
    voltage = float(psu.read_voltage(1))
    assert voltage == 5.0
    print(f"{identity}: read back {voltage:g} V")
finally:
    psu.close()

## Conclusion

Start with the commands you need, set channel bounds, and test each mapping in simulation.

In [ ]:
scratch.cleanup()